# P2-G4 Local Agent 1: P4 Data Contract Audit

이 노트북은 모델 학습을 하지 않는다. P2-G3 `p4_handoff_candidate/shared/P4_HANDOFF_MANIFEST.json`가 가리키는 candidate 데이터와 bridge, manifest, registry를 독립적으로 다시 계산해 P4가 읽을 데이터 계약을 확정한다.

원본 P3 candidate 파일과 P3 노트북은 수정하지 않는다. 모든 새 산출물은 `workbook/p2/p2_4/p4_1_data_contract/` 아래에만 저장한다.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for path in [cur, *cur.parents]:
        if (path / "scripts" / "p2_g4_1_data_contract_local1.py").exists():
            return path
    raise RuntimeError("repo root not found")


ROOT = find_repo_root(Path.cwd())
SCRIPT = ROOT / "scripts" / "p2_g4_1_data_contract_local1.py"
OUT_ROOT = ROOT / "workbook" / "p2" / "p2_4" / "p4_1_data_contract"
QA = OUT_ROOT / "qa"
PROFILES = OUT_ROOT / "profiles"
SAMPLES = OUT_ROOT / "samples"
LOGS = OUT_ROOT / "logs"
REPORTS = OUT_ROOT / "reports"
ARTIFACTS = OUT_ROOT / "artifacts"

print("ROOT:", ROOT)
print("SCRIPT:", SCRIPT)
print("OUT_ROOT:", OUT_ROOT)


ROOT: /home/sieg/projects-wsl/SBS_dataScience
SCRIPT: /home/sieg/projects-wsl/SBS_dataScience/scripts/p2_g4_1_data_contract_local1.py
OUT_ROOT: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract


## Gate Run - Execute Contract Script


In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout[-8000:])
if result.stderr.strip():
    print(result.stderr[-8000:])
if result.returncode != 0:
    raise RuntimeError(f"contract script failed: {result.returncode}")

summary = json.loads((REPORTS / "final_status.json").read_text(encoding="utf-8"))
display(pd.DataFrame([
    {
        "final_status": summary["final_status"],
        "fail_count": summary["fail_count"],
        "warn_count": summary["warn_count"],
        "output_root": summary["outputs"]["root"],
    }
]))


: 6366,
        "actual_with_structure_n": 6366,
        "registry_train_n": 5499,
        "actual_train_n": 5499,
        "registry_val_n": 1141,
        "actual_val_n": 1141,
        "registry_test_n": 858,
        "actual_test_n": 858,
        "registry_actual_mismatch_n": 0,
        "status": "PASS"
      },
      {
        "sample_id": "JOINT_EMP_PROG",
        "registry_usable_rows_n": 7389,
        "actual_usable_rows_n": 7389,
        "registry_with_structure_n": 6270,
        "actual_with_structure_n": 6270,
        "registry_train_n": 5428,
        "actual_train_n": 5428,
        "registry_val_n": 1121,
        "actual_val_n": 1121,
        "registry_test_n": 840,
        "actual_test_n": 840,
        "registry_actual_mismatch_n": 0,
        "status": "PASS"
      }
    ]
  },
  "gate7": {
    "registry_missing_n": 0,
    "all_null_active_feature_set_n": 0,
    "p4_use_column_n": 120,
    "target_candidate_n": 6,
    "feature_set_n": 4
  },
  "paths": {
    "D01_v2": "/home/s

,final_status,fail_count,warn_count,output_root
0,P4_DATA_CONTRACT_READY,0,1,/home/sieg/projects-wsl/SBS_dataScience/workbo...


In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)


def show_csv(title: str, path: Path, n: int | None = None) -> pd.DataFrame:
    display(Markdown(f"### {title}\n`{path}`"))
    df = read_csv(path)
    display(df if n is None else df.head(n))
    print("shape:", df.shape)
    return df


def show_md(title: str, path: Path) -> None:
    display(Markdown(f"### {title}\n`{path}`"))
    display(Markdown(path.read_text(encoding="utf-8")))


def gate_summary(gate_name: str, payload: dict) -> None:
    display(Markdown(f"## {gate_name} Summary"))
    display(pd.DataFrame([payload]))


## Gate 0 - Manifest And Lineage Integrity


In [ ]:
gate_summary("Gate 0", summary["gate0"])
manifest_audit = show_csv("Manifest integrity audit", QA / "manifest_integrity_audit.csv")
display(manifest_audit[["artifact_name", "section", "path", "manifest_hash", "actual_hash", "hash_match", "manifest_shape", "actual_shape", "shape_match", "modified_after_manifest", "status"]])
display(manifest_audit["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows"))


## Gate 0 Summary

,manifest_audit_rows,hash_mismatch_n,shape_mismatch_n,missing_n,modified_after_manifest_n,hard_fail_n
0,37,0,0,0,0,0


### Manifest integrity audit
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/manifest_integrity_audit.csv`

,artifact_name,section,path,manifest_hash,actual_hash,hash_match,manifest_shape,actual_shape,shape_match,manifest_created_at,actual_mtime,modified_after_manifest,status
0,P4_HANDOFF_MANIFEST,manifest_self,/home/sieg/projects-wsl/SBS_dataScience/workbo...,b9ea9528b24630f15be8913012b40df81692d16e849120...,b9ea9528b24630f15be8913012b40df81692d16e849120...,True,NaN,NaN,True,2026-07-12T21:47:51,2026-07-12T21:47:51,False,PASS
1,raw_excel,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,c767832c42fb2be505c4d9bb8ae0bc39891091f86c6ae8...,c767832c42fb2be505c4d9bb8ae0bc39891091f86c6ae8...,True,NaN,NaN,True,2026-07-12T21:47:51,2026-07-12T16:54:55,False,PASS
2,outcome_spine_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2530f88a0d923edf11c83e5884d2bb47d1811679cc9b61...,2530f88a0d923edf11c83e5884d2bb47d1811679cc9b61...,True,NaN,"[10242, 17]",True,2026-07-12T21:47:51,2026-07-12T19:12:08,False,PASS
3,wage_reference_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,90f94f0b781d88a45a168a3b60095b16537a532de1dbee...,90f94f0b781d88a45a168a3b60095b16537a532de1dbee...,True,NaN,"[14, 66]",True,2026-07-12T21:47:51,2026-07-12T19:12:08,False,PASS
4,wage_quartile_reference_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,c46a158b418c8ec114af62a95d978692cc75890ff55006...,c46a158b418c8ec114af62a95d978692cc75890ff55006...,True,NaN,"[23, 10]",True,2026-07-12T21:47:51,2026-07-12T19:12:08,False,PASS
5,wage_column_contract_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2431de62f6b0b8d3276b603d64f94c7cbdeaa4450aa79f...,2431de62f6b0b8d3276b603d64f94c7cbdeaa4450aa79f...,True,NaN,"[66, 10]",True,2026-07-12T21:47:51,2026-07-12T19:12:08,False,PASS
6,job_cert_bridge_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,e88fe083e07430f96f5549071cde674021048c3f482f69...,e88fe083e07430f96f5549071cde674021048c3f482f69...,True,NaN,"[24, 26]",True,2026-07-12T21:47:51,2026-07-12T19:12:09,False,PASS
7,D01_v2,output_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2f187b5af44c828d4107af12368029caf6b83b6254af75...,2f187b5af44c828d4107af12368029caf6b83b6254af75...,True,"[34969, 186]","[34969, 186]",True,2026-07-12T21:47:51,2026-07-12T21:47:51,False,PASS
8,D02_v2,output_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,45f8aa40f31e14b83b5d97f594abf89bb4a5aaa4bc6773...,45f8aa40f31e14b83b5d97f594abf89bb4a5aaa4bc6773...,True,"[10242, 37]","[10242, 37]",True,2026-07-12T21:47:51,2026-07-12T21:47:51,False,PASS
9,D03_v2,output_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,c6fd569052684502e5bab5758510d3cd945f68ddeaa47f...,c6fd569052684502e5bab5758510d3cd945f68ddeaa47f...,True,"[10242, 108]","[10242, 108]",True,2026-07-12T21:47:51,2026-07-12T21:47:51,False,PASS


shape: (37, 13)


,artifact_name,section,path,manifest_hash,actual_hash,hash_match,manifest_shape,actual_shape,shape_match,modified_after_manifest,status
0,P4_HANDOFF_MANIFEST,manifest_self,/home/sieg/projects-wsl/SBS_dataScience/workbo...,b9ea9528b24630f15be8913012b40df81692d16e849120...,b9ea9528b24630f15be8913012b40df81692d16e849120...,True,NaN,NaN,True,False,PASS
1,raw_excel,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,c767832c42fb2be505c4d9bb8ae0bc39891091f86c6ae8...,c767832c42fb2be505c4d9bb8ae0bc39891091f86c6ae8...,True,NaN,NaN,True,False,PASS
2,outcome_spine_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2530f88a0d923edf11c83e5884d2bb47d1811679cc9b61...,2530f88a0d923edf11c83e5884d2bb47d1811679cc9b61...,True,NaN,"[10242, 17]",True,False,PASS
3,wage_reference_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,90f94f0b781d88a45a168a3b60095b16537a532de1dbee...,90f94f0b781d88a45a168a3b60095b16537a532de1dbee...,True,NaN,"[14, 66]",True,False,PASS
4,wage_quartile_reference_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,c46a158b418c8ec114af62a95d978692cc75890ff55006...,c46a158b418c8ec114af62a95d978692cc75890ff55006...,True,NaN,"[23, 10]",True,False,PASS
5,wage_column_contract_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2431de62f6b0b8d3276b603d64f94c7cbdeaa4450aa79f...,2431de62f6b0b8d3276b603d64f94c7cbdeaa4450aa79f...,True,NaN,"[66, 10]",True,False,PASS
6,job_cert_bridge_raw,source_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,e88fe083e07430f96f5549071cde674021048c3f482f69...,e88fe083e07430f96f5549071cde674021048c3f482f69...,True,NaN,"[24, 26]",True,False,PASS
7,D01_v2,output_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,2f187b5af44c828d4107af12368029caf6b83b6254af75...,2f187b5af44c828d4107af12368029caf6b83b6254af75...,True,"[34969, 186]","[34969, 186]",True,False,PASS
8,D02_v2,output_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,45f8aa40f31e14b83b5d97f594abf89bb4a5aaa4bc6773...,45f8aa40f31e14b83b5d97f594abf89bb4a5aaa4bc6773...,True,"[10242, 37]","[10242, 37]",True,False,PASS
9,D03_v2,output_files,/home/sieg/projects-wsl/SBS_dataScience/workbo...,c6fd569052684502e5bab5758510d3cd945f68ddeaa47f...,c6fd569052684502e5bab5758510d3cd945f68ddeaa47f...,True,"[10242, 108]","[10242, 108]",True,False,PASS


,status,rows
0,PASS,37


## Gate 1 - D01 v2 Grain And Count Conservation


In [ ]:
gate_summary("Gate 1", summary["gate1"])
display(Markdown(f"**Grain sentence:** {summary['gate1']['d01_sentence']}"))
show_csv("D01 dtype and key profile", PROFILES / "d01_grain_profile.csv")
show_csv("D01 top missing columns", PROFILES / "d01_missing_top200.csv", n=40)
show_csv("D01 count conservation", QA / "d01_count_conservation.csv")
show_csv("D01 duplicate audit", QA / "d01_duplicate_audit.csv")
show_csv("D01 row-axis comparison", PROFILES / "d01_row_axis_comparison.csv", n=120)
show_md("D01 grain explanation", REPORTS / "D01_GRAIN_EXPLANATION.md")


## Gate 1 Summary

,d01_shape,raw_excel_shape,old_main_shape,d01_key,d01_key_nunique,d01_key_duplicate_n,d01_exact_duplicate_raw_columns_n,d01_lineage_missing_n,d01_unexplained_count_delta_n,d01_sentence
0,"[34969, 186]","[34969, 130]","[15727, 120]","[analysis_year, school_name_std, campus_name_s...",34969,0,0,0,0,D01의 한 행은 2024년 × 학교표준명 × 캠퍼스(본분교/제N캠퍼스) × 주야구...


**Grain sentence:** D01의 한 행은 2024년 × 학교표준명 × 캠퍼스(본분교/제N캠퍼스) × 주야구분 × 학위과정 × KEDI 학과코드의 구조자료 관측치이다.

### D01 dtype and key profile
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/profiles/d01_grain_profile.csv`

,dataset,dtype,column_count,rows,columns,key_columns,key_duplicate_rows,full_duplicate_rows,missing_cells
0,D01_v2,int64,109,34969,186,analysis_year|school_name_std|campus_name_std|...,0,0,144867
1,D01_v2,string,26,34969,186,analysis_year|school_name_std|campus_name_std|...,0,0,144867
2,D01_v2,str,24,34969,186,analysis_year|school_name_std|campus_name_std|...,0,0,144867
3,D01_v2,Int32,13,34969,186,analysis_year|school_name_std|campus_name_std|...,0,0,144867
4,D01_v2,Float32,10,34969,186,analysis_year|school_name_std|campus_name_std|...,0,0,144867
5,D01_v2,boolean,3,34969,186,analysis_year|school_name_std|campus_name_std|...,0,0,144867
6,D01_v2,Int16,1,34969,186,analysis_year|school_name_std|campus_name_std|...,0,0,144867


...

### D01 row-axis comparison
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/profiles/d01_row_axis_comparison.csv`

,axis,value,raw_excel_rows,old_15727_csv_rows,d01_rows
0,학위과정,대학과정,15526,7494,15526
1,학위과정,대학원과정,12575,7467,12575
2,학위과정,전문대학과정,6868,766,6868
3,주야구분,계절제,155,0,155
4,주야구분,야간,3943,0,3943
5,주야구분,야간+계절제,244,0,244
6,주야구분,야간+원격,18,0,18
7,주야구분,원격,618,0,618
8,주야구분,주간,15732,0,15732
9,주야구분,주간+계절제,46,0,46


shape: (48, 5)


### D01 grain explanation
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/reports/D01_GRAIN_EXPLANATION.md`

# D01 Grain Explanation

D01의 한 행은 2024년 × 학교표준명 × 캠퍼스(본분교/제N캠퍼스) × 주야구분 × 학위과정 × KEDI 학과코드의 구조자료 관측치이다.

## 실제 행수 해석
- raw Excel sheet `학교별 학과별 주요 현황`에서 빈 행과 대계열 결측 행을 제거하면 **34,969행**이다.
- 기존 15,727행 CSV는 이 raw Excel을 모두 담은 원천이 아니라, 주야구분·학교상태 등 일부 축이 빠진 prefiltered 중간 산출물이다.
- D01 v2는 15,727행을 복제해 늘린 것이 아니라, raw Excel의 34,969행을 다시 읽어 `주야구분`, `학교상태`, `학위과정`, `본분교` 축을 보존한 구조 master다.

## 확정 grain
- key: `analysis_year|school_name_std|campus_name_std|day_evening_raw|degree_course|kedi_dept_code`
- canonical grain duplicate rows: `0`
- lineage missing rows: `0`

## Count conservation
- raw Excel 대비 D01 count 합계 불일치 metric 수: `0`
- 15,727행 CSV 대비 차이는 범위 복구(scope restoration)로 별도 기록했다.


## Gate 2 - D03/D08 Spine Integrity


In [ ]:
gate_summary("Gate 2", summary["gate2"])
show_csv("D03/D08 spine comparison", QA / "d03_d08_spine_comparison.csv")
show_csv("P4 spine key contract", ARTIFACTS / "p4_spine_key_contract.csv")
show_csv("D08 P4 composite key duplicate rows", QA / "d08_composite_key_duplicates.csv")
display(Markdown("### D08 normalized-name duplicate rows (punctuation-normalized raw names)"))
norm_dups = read_csv(QA / "d08_normalized_name_duplicate_rows.csv")
display(norm_dups)
print("shape:", norm_dups.shape)
display(Markdown("### D08 carried headcount UID duplicate rows"))
hc_dups = read_csv(QA / "d08_headcount_uid_duplicate_rows.csv")
display(hc_dups)
print("shape:", hc_dups.shape)
show_csv("D08 rate range audit", QA / "d08_rate_range_audit.csv")


## Gate 2 Summary

,d03_shape,d08_shape,p4_composite_key_duplicate_n,headcount_uid_composite_duplicate_n,normalized_name_duplicate_pair_rows,outcome_value_mismatch_n,rate_range_violation_n,inf_n
0,"[10242, 108]","[10242, 151]",0,18,8,0,0,0


### D03/D08 spine comparison
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/d03_d08_spine_comparison.csv`

,check,actual,expected,status
0,D03 row count,10242,10242,PASS
1,D08 row count,10242,10242,PASS
2,D03 outcome_row_id duplicate,0,0,PASS
3,D08 outcome_row_id duplicate,0,0,PASS
4,P4 raw outcome composite key duplicate,0,0,PASS
5,D08 carried headcount UID composite duplicate,18,tracked as WARN only,WARN
6,D03 to D08 lost rows,0,0,PASS
7,D08 extra rows over D03,0,0,PASS
8,D03-D08 order changed,False,False,PASS
9,D03-D08 outcome value mismatch cells,0,0,PASS


shape: (12, 4)


...

shape: (36, 9)


### D08 rate range audit
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/d08_rate_range_audit.csv`

,column,non_null_n,min,max,range_violation_n,status
0,selectivity_proxy_pct,3737,0.1,100.000000,0,PASS
1,a_rate_pct,10242,0.0,100.000000,0,PASS
2,cd_rate_pct,10242,0.0,100.000000,0,PASS
3,f_rate_pct,10242,0.0,100.000000,0,PASS
4,employment_rate_pct,7477,0.0,100.000000,0,PASS
5,health_employment_rate_pct,7477,0.0,100.000000,0,PASS
6,progression_rate_pct,7587,0.0,100.000000,0,PASS
7,vocational_college_progression_rate_pct,7587,0.0,50.000000,0,PASS
8,university_progression_rate_pct,7587,0.0,25.000000,0,PASS
9,graduate_school_progression_rate_pct,7587,0.0,100.000000,0,PASS


shape: (12, 6)


,column,non_null_n,min,max,range_violation_n,status
0,selectivity_proxy_pct,3737,0.1,100.000000,0,PASS
1,a_rate_pct,10242,0.0,100.000000,0,PASS
2,cd_rate_pct,10242,0.0,100.000000,0,PASS
3,f_rate_pct,10242,0.0,100.000000,0,PASS
4,employment_rate_pct,7477,0.0,100.000000,0,PASS
5,health_employment_rate_pct,7477,0.0,100.000000,0,PASS
6,progression_rate_pct,7587,0.0,100.000000,0,PASS
7,vocational_college_progression_rate_pct,7587,0.0,50.000000,0,PASS
8,university_progression_rate_pct,7587,0.0,25.000000,0,PASS
9,graduate_school_progression_rate_pct,7587,0.0,100.000000,0,PASS


## Gate 3 - Structure Match Verification


In [ ]:
gate_summary("Gate 3", summary["gate3"])
show_csv("Structure match integrity", QA / "structure_match_integrity.csv")
show_csv("match_method distribution", PROFILES / "structure_match_method_distribution.csv")
show_csv("Structure feature non-null by match_method", PROFILES / "structure_feature_non_null_by_match_method.csv")
show_csv("split x match_method", PROFILES / "split_by_match_method_crosstab.csv")
show_csv("major_group_7 x match_method", PROFILES / "major_group_7_by_match_method_crosstab.csv")
structure_sample = show_csv("Structure stratified sample seed=3085", SAMPLES / "structure_match_sample_seed3085.csv", n=120)
display(structure_sample["match_method"].value_counts(dropna=False).rename_axis("match_method").reset_index(name="sample_rows"))


## Gate 3 Summary

,match_method_distribution,high_conf_match_n,high_conf_match_rate,auto_candidate_ge2_n,auto_campus_conflict_n,structure_context_mismatch_n
0,"{'exact_strict': 8556, 'manual_pending': 1316,...",8561,0.835872,0,0,0


### Structure match integrity
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/structure_match_integrity.csv`

,check,actual,expected,status
0,bridge row count,10242,10242,PASS
1,bridge outcome_row_id duplicate,0,0,PASS
2,candidate_count>=2 auto-confirmed,0,0,PASS
3,campus conflict auto-confirmed,0,0,PASS
4,manual_pending included in high confidence,0,0,PASS
5,unmatched rows deleted from D08,0,0,PASS
6,structural_match_eligible formula mismatch,0,0,PASS


shape: (7, 4)


...

shape: (8, 5)


### Structure stratified sample seed=3085
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/samples/structure_match_sample_seed3085.csv`

,outcome_row_id,school_name_raw,campus_name_raw_x,dept_name_raw,campus_name_std_x,dept_norm_strict,dept_norm_suffix_reduced,headcount_row_id,campus_name_raw_y,campus_name_std_y,field_middle_name,field_small_name,candidate_count,match_method,match_status,match_score,match_evidence,candidate_preview,admission_capacity_n,recruitment_n,applicants_n,admits_n,enrolled_students_n,graduates_n
0,OC2024_03902,동덕여자대학교,NaN,미술학부 디지털공예전공,unknown,미술학부디지털공예전공,미술학부디지털공예,HC2_2024_13537,본교(제1캠퍼스),본교_제1캠퍼스,응용예술,공예,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,미술학부 디지털공예전공,0.0,35.0,920.0,35.0,61.0,0.0
1,OC2024_06496,신라대학교,NaN,전기전자공학과,unknown,전기전자공학과,전기전자공,HC2_2024_18218,본교(제1캠퍼스),본교_제1캠퍼스,전기ㆍ전자,전자공학,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,전기전자공학과,20.0,22.0,129.0,11.0,56.0,0.0
2,OC2024_08182,제주국제대학교,NaN,건축학과,unknown,건축학과,건축,HC2_2024_26140,본교(제1캠퍼스),본교_제1캠퍼스,건축,건축학,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,건축학과,0.0,0.0,0.0,0.0,37.0,23.0
3,OC2024_02892,극동대학교,NaN,호텔경영학과,unknown,호텔경영학과,호텔경영,HC2_2024_11314,본교(제1캠퍼스),본교_제1캠퍼스,경영ㆍ경제,경영학,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,호텔경영학과,106.0,154.0,446.0,134.0,161.0,0.0
4,OC2024_06623,아주대학교,NaN,물리학과,unknown,물리학과,물리,HC2_2024_18446,본교(제1캠퍼스),본교_제1캠퍼스,수학ㆍ물리ㆍ천문ㆍ지리,물리ㆍ과학,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,물리학과,33.0,35.0,274.0,35.0,215.0,30.0
5,OC2024_04452,동의대학교,NaN,환경공학과,unknown,환경공학과,환경공,HC2_2024_14384,본교(제1캠퍼스),본교_제1캠퍼스,생물ㆍ화학ㆍ환경,환경학,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,환경공학과,35.0,36.0,113.0,34.0,35.0,0.0
6,OC2024_08436,중부대학교,NaN,전자출판인쇄공학전공,unknown,전자출판인쇄공학전공,전자출판인쇄공학,HC2_2024_21507,본교(제2캠퍼스),본교_제2캠퍼스,기타,응용공학,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,전자출판인쇄공학전공,0.0,0.0,0.0,0.0,22.0,12.0
7,OC2024_07326,원광대학교,NaN,스포츠과학부,unknown,스포츠과학부,스포츠과,HC2_2024_20008,본교(제1캠퍼스),본교_제1캠퍼스,무용ㆍ체육,체육,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,스포츠과학부,120.0,120.0,456.0,119.0,749.0,107.0
8,OC2024_06965,영산대학교(양산)_제2캠퍼스,제2캠퍼스|양산,건축공학과,제2캠퍼스,건축공학과,건축공,HC2_2024_25422,본교(제2캠퍼스),본교_제2캠퍼스,건축,건축ㆍ설비공학,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,건축공학과,0.0,0.0,0.0,0.0,48.0,10.0
9,OC2024_09423,한국항공대학교,NaN,항공전자정보공학부,unknown,항공전자정보공학부,항공전자정보공,HC2_2024_22807,본교(제1캠퍼스),본교_제1캠퍼스,컴퓨터ㆍ통신,정보ㆍ통신공학,1,exact_strict,auto_high_confidence,1.0,school+campus+department strict exact unique,항공전자정보공학부,0.0,0.0,0.0,0.0,1087.0,207.0


shape: (205, 24)


,match_method,sample_rows
0,manual_pending,100
1,exact_strict,50
2,unmatched,50
3,exact_token_unique,5


## Gate 4 - major_group_7 Verification


In [ ]:
gate_summary("Gate 4", summary["gate4"])
show_csv("Major mapping integrity", QA / "major7_mapping_integrity.csv")
show_csv("major7 mapping method distribution", PROFILES / "major7_mapping_method_distribution.csv")
show_csv("major_group_7 distribution", PROFILES / "major_group_7_distribution.csv")
show_csv("split x major_group_7", PROFILES / "split_by_major_group_7_crosstab.csv")
show_csv("mapping_method x major_group_7", PROFILES / "major_mapping_method_by_major_group_7_crosstab.csv")
show_csv("mapping confidence by method", PROFILES / "major_mapping_confidence_by_method.csv")
special = show_csv("Special major term search", QA / "major7_special_term_search.csv", n=200)
display(special.groupby(["search_term", "major7_mapping_method", "major_group_7"], dropna=False).size().reset_index(name="rows"))
major_sample = show_csv("major7 stratified sample", SAMPLES / "major7_mapping_sample.csv", n=160)
display(major_sample["major7_mapping_method"].value_counts(dropna=False).rename_axis("major7_mapping_method").reset_index(name="sample_rows"))


## Gate 4 Summary

,major_method_distribution,major_group_distribution,major_high_medium_n,major_high_medium_rate,ambiguous_unknown_n,ambiguous_unknown_rate,major_allowed_code_violation_n,inherited_major_mismatch_n
0,"{'inherited_headcount': 8561, 'keyword_rule': ...","{'ENG': 2642, 'SOC': 2165, 'ART': 1566, 'NAT':...",10099,0.986038,143,0.013962,0,0


### Major mapping integrity
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/major7_mapping_integrity.csv`

,check,actual,expected,status
0,allowed major_group_7 code violations,0.000000,0,PASS
1,major high/medium rate >=97%,0.986038,>=0.97,PASS
2,ambiguous+unknown <=3%,0.013962,<=0.03,PASS
3,inherited D01 major mismatch,0.000000,0,PASS
4,major_context_eligible formula mismatch,0.000000,0,PASS
5,ambiguous/unknown rows included in major samples,0.000000,0,PASS


shape: (6, 4)


...

,search_term,major7_mapping_method,major_group_7,rows
0,AI,ambiguous,NaN,1
1,AI,exact_dictionary,ENG,8
2,AI,inherited_headcount,ART,4
3,AI,inherited_headcount,ENG,109
4,AI,inherited_headcount,NAT,2
5,AI,inherited_headcount,SOC,5
6,AI,keyword_rule,ART,1
7,AI,keyword_rule,ENG,3
8,AI,keyword_rule,MED,2
9,AI,keyword_rule,SOC,1


### major7 stratified sample
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/samples/major7_mapping_sample.csv`

,outcome_row_id,major7_mapping_method,major7_mapping_confidence,dept_name_raw,major7_evidence,split,major_group_7
0,OC2024_05252,inherited_headcount,high,미용예술학과,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
1,OC2024_05081,inherited_headcount,high,인더스트리얼디자인전공,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
2,OC2024_01405,inherited_headcount,high,PostModern음악학과,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
3,OC2024_00243,inherited_headcount,high,스포츠복지학과,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
4,OC2024_09660,inherited_headcount,high,동양화전공,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
5,OC2024_08409,inherited_headcount,high,레저스포츠학전공,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
6,OC2024_03876,inherited_headcount,high,관현악과,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
7,OC2024_09214,inherited_headcount,high,미디어영상학과,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
8,OC2024_05577,inherited_headcount,high,뷰티학과,D01 v2 high-confidence match 대계열=예체능계열,train,NaN
9,OC2024_07047,inherited_headcount,high,음악전공,D01 v2 high-confidence match 대계열=예체능계열,train,NaN


shape: (541, 7)


,major7_mapping_method,sample_rows
0,keyword_rule,188
1,exact_dictionary,140
2,ambiguous,85
3,inherited_headcount,70
4,unknown,58


## Gate 5 - Local 2 GOMS Context Lineage


In [ ]:
gate_summary("Gate 5", summary["gate5"])
show_csv("Local2 context lineage integrity", QA / "local2_context_lineage_integrity.csv")
show_csv("GOMS D07->D08 exact comparison", QA / "goms_d07_d08_exact_comparison.csv")
show_csv("GOMS context unique values by major", PROFILES / "goms_context_unique_by_major.csv")


## Gate 5 Summary

,d07_path,d07_hash,d07_shape,d07_ready_for_local1,d07_key_duplicate_n,d07_d08_goms_mismatch_n
0,/home/sieg/projects-wsl/SBS_dataScience/workbo...,f44c6f9e7a7539361dce301686144780e0ce588deb929b...,"[7, 29]",True,0,0


### Local2 context lineage integrity
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/local2_context_lineage_integrity.csv`

,check,actual,expected,status
0,Local2 ready_for_local1,True,True,PASS
1,manifest D07 hash equals actual,f44c6f9e7a7539361dce301686144780e0ce588deb929b...,f44c6f9e7a7539361dce301686144780e0ce588deb929b...,PASS
2,actual D07 hash equals Local1 manifest input,f44c6f9e7a7539361dce301686144780e0ce588deb929b...,f44c6f9e7a7539361dce301686144780e0ce588deb929b...,PASS
3,D07 row count,7,7,PASS
4,D07 major key duplicate,0,0,PASS
5,D07 to D08 goms exact mismatch cells,0,0,PASS


shape: (6, 4)


...

shape: (28, 4)


### GOMS context unique values by major
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/profiles/goms_context_unique_by_major.csv`

,major_group_7,d08_rows,goms_profile_start_year_d08_unique_n,goms_profile_end_year_d08_unique_n,goms_profile_years_n_d08_unique_n,goms_aggregation_method_d08_unique_n,goms_recent_employment_rate_pct_d08_unique_n,goms_recent_firm_300plus_pct_d08_unique_n,goms_recent_public_nonprofit_pct_d08_unique_n,goms_recent_permanent_pct_d08_unique_n,goms_recent_unstable_pct_d08_unique_n,goms_recent_self_employed_pct_d08_unique_n,goms_recent_industry_hhi_d08_unique_n,goms_recent_industry_top3_pct_d08_unique_n,goms_recent_professional_highskill_pct_d08_unique_n,goms_recent_mean_income_10kkrw_d08_unique_n,goms_recent_weekly_work_hours_d08_unique_n,goms_recent_hourly_income_proxy_d08_unique_n,goms_income_trend_per_year_d08_unique_n,goms_hours_trend_per_year_d08_unique_n,goms_firm_300plus_trend_per_year_d08_unique_n,goms_permanent_trend_per_year_d08_unique_n,goms_latest_2019_mean_income_10kkrw_d08_unique_n,goms_latest_2019_weekly_work_hours_d08_unique_n,goms_latest_2019_firm_300plus_pct_d08_unique_n,goms_latest_2019_permanent_pct_d08_unique_n,goms_source_years_observed_d08_unique_n,goms_year_over_year_review_flag_d08_unique_n,goms_mapping_confidence_d08_unique_n,goms_row_qa_status_d08_unique_n
0,ART,1566,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
1,EDU,728,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2,ENG,2642,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
3,HUM,1108,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
4,MED,632,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
5,NAT,1258,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
6,SOC,2165,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
7,NaN,143,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


shape: (8, 30)


,major_group_7,d08_rows,goms_profile_start_year_d08_unique_n,goms_profile_end_year_d08_unique_n,goms_profile_years_n_d08_unique_n,goms_aggregation_method_d08_unique_n,goms_recent_employment_rate_pct_d08_unique_n,goms_recent_firm_300plus_pct_d08_unique_n,goms_recent_public_nonprofit_pct_d08_unique_n,goms_recent_permanent_pct_d08_unique_n,goms_recent_unstable_pct_d08_unique_n,goms_recent_self_employed_pct_d08_unique_n,goms_recent_industry_hhi_d08_unique_n,goms_recent_industry_top3_pct_d08_unique_n,goms_recent_professional_highskill_pct_d08_unique_n,goms_recent_mean_income_10kkrw_d08_unique_n,goms_recent_weekly_work_hours_d08_unique_n,goms_recent_hourly_income_proxy_d08_unique_n,goms_income_trend_per_year_d08_unique_n,goms_hours_trend_per_year_d08_unique_n,goms_firm_300plus_trend_per_year_d08_unique_n,goms_permanent_trend_per_year_d08_unique_n,goms_latest_2019_mean_income_10kkrw_d08_unique_n,goms_latest_2019_weekly_work_hours_d08_unique_n,goms_latest_2019_firm_300plus_pct_d08_unique_n,goms_latest_2019_permanent_pct_d08_unique_n,goms_source_years_observed_d08_unique_n,goms_year_over_year_review_flag_d08_unique_n,goms_mapping_confidence_d08_unique_n,goms_row_qa_status_d08_unique_n
0,ART,1566,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
1,EDU,728,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2,ENG,2642,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
3,HUM,1108,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
4,MED,632,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
5,NAT,1258,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
6,SOC,2165,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
7,NaN,143,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


## Gate 6 - Split And Sample Registry


In [ ]:
gate_summary("Gate 6", summary["gate6"])
show_csv("Split/sample integrity", QA / "split_sample_integrity.csv")
show_csv("Split row/school summary", PROFILES / "split_row_school_summary.csv")
show_csv("Sample registry actual check", QA / "sample_registry_actual_check.csv")
target_missing = show_csv("Target missing by split and major", PROFILES / "target_missing_by_split_major.csv", n=120)
feature_missing = show_csv("Feature missing by sample", PROFILES / "feature_missing_by_sample.csv", n=120)
display(feature_missing.groupby("sample_id", dropna=False).agg(feature_columns=("feature_column", "nunique"), mean_missing_rate=("missing_rate", "mean")).reset_index())


## Gate 6 Summary

,split_leakage_n,split_summary,sample_registry_mismatch_rows,sample_actual
0,0,"[{'split': 'test', 'rows': 1199, 'schools': 30...",0,"[{'sample_id': 'GRADE_ALL', 'registry_usable_r..."


### Split/sample integrity
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/split_sample_integrity.csv`

,check,actual,expected,status
0,school split leakage,0,0,PASS
1,membership row count,10242,10242,PASS
2,membership outcome_row_id duplicate,0,0,PASS
3,sample registry actual mismatch rows,0,0,PASS


shape: (4, 4)


...

shape: (96, 7)


### Feature missing by sample
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/profiles/feature_missing_by_sample.csv`

,sample_id,feature_column,rows,missing_n,missing_rate
0,GRADE_ALL,ctx24_reference_sample_n,10099,0,0.000000
1,GRADE_ALL,ctx24_mean_income_10kkrw,10099,0,0.000000
2,GRADE_ALL,ctx24_median_income_10kkrw,10099,0,0.000000
3,GRADE_ALL,ctx24_income_300plus_pct,10099,0,0.000000
4,GRADE_ALL,ctx24_income_400plus_pct,10099,0,0.000000
5,GRADE_ALL,ctx24_large_company_pct,10099,0,0.000000
6,GRADE_ALL,ctx24_mid_company_pct,10099,0,0.000000
7,GRADE_ALL,ctx24_small_company_pct,10099,0,0.000000
8,GRADE_ALL,ctx24_large_mid_company_pct,10099,0,0.000000
9,GRADE_ALL,ctx24_public_nonprofit_pct,10099,0,0.000000


shape: (330, 5)


,sample_id,feature_columns,mean_missing_rate
0,EMPLOYMENT_HEALTH,66,0.105530
1,GRADE_ALL,66,0.105328
2,GRADE_SELECTIVITY,66,0.079685
3,JOINT_EMP_PROG,66,0.105530
4,PROGRESSION_GRADSCHOOL,66,0.105372


## Gate 7 - Registry And Feature Set Contract


In [ ]:
gate_summary("Gate 7", summary["gate7"])
show_csv("Registry integrity", QA / "registry_integrity.csv")
show_csv("All-null feature audit", QA / "all_null_feature_audit.csv")
show_csv("P4 feature set registry", ROOT / "workbook" / "p2" / "p2_3" / "p4_handoff_candidate" / "shared" / "p4_feature_set_registry.csv")
show_csv("P4 target candidate registry", ROOT / "workbook" / "p2" / "p2_3" / "p4_handoff_candidate" / "shared" / "p4_target_candidate_registry.csv")
show_csv("D08 column registry dtype profile", PROFILES / "d08_v2_dtype_profile.csv")
show_csv("D08 top missing columns", PROFILES / "d08_v2_missing_top200.csv", n=80)


## Gate 7 Summary

,registry_missing_n,all_null_active_feature_set_n,p4_use_column_n,target_candidate_n,feature_set_n
0,0,0,120,6,4


### Registry integrity
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/registry_integrity.csv`

,check,actual,expected,status
0,D08 columns missing from column registry,0,0,PASS
1,p4_use registry columns missing from D08,0,0,PASS
2,target registry columns missing from D08,0,0,PASS
3,unknown sample ids,0,0,PASS
4,all-null active feature sets,0,0,PASS


shape: (5, 4)


...

shape: (10, 9)


### D08 top missing columns
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/profiles/d08_v2_missing_top200.csv`

,dataset,column,missing_n,missing_rate,dtype
0,D08_v2,ctx24_industry_top3_pct,10242,1.000000,Float32
1,D08_v2,ctx24_industry_hhi,10242,1.000000,Float32
2,D08_v2,selectivity_proxy_pct,6505,0.635130,Float32
3,D08_v2,competition_ratio,4985,0.486721,Float32
4,D08_v2,admission_yield_ratio,4985,0.486721,Float32
5,D08_v2,admit_per_applicant_ratio,4982,0.486428,Float32
6,D08_v2,student_faculty_ratio,3863,0.377172,Float32
7,D08_v2,fulltime_faculty_share_pct,3863,0.377172,Float32
8,D08_v2,employment_rate_pct,2765,0.269967,Float32
9,D08_v2,health_employment_rate_pct,2765,0.269967,Float32


shape: (151, 5)


,dataset,column,missing_n,missing_rate,dtype
0,D08_v2,ctx24_industry_top3_pct,10242,1.000000,Float32
1,D08_v2,ctx24_industry_hhi,10242,1.000000,Float32
2,D08_v2,selectivity_proxy_pct,6505,0.635130,Float32
3,D08_v2,competition_ratio,4985,0.486721,Float32
4,D08_v2,admission_yield_ratio,4985,0.486721,Float32
5,D08_v2,admit_per_applicant_ratio,4982,0.486428,Float32
6,D08_v2,student_faculty_ratio,3863,0.377172,Float32
7,D08_v2,fulltime_faculty_share_pct,3863,0.377172,Float32
8,D08_v2,employment_rate_pct,2765,0.269967,Float32
9,D08_v2,health_employment_rate_pct,2765,0.269967,Float32


## Reports And Execution Evidence


In [ ]:
show_md("P4 D08 data card", REPORTS / "P4_D08_DATA_CARD.md")
show_md("P4 data integrity report", REPORTS / "P4_DATA_INTEGRITY_REPORT.md")
show_csv("All gate QA summary", QA / "all_gate_qa_summary.csv")
show_csv("Failed checks", QA / "failed_checks.csv")
show_csv("Warn checks", QA / "warn_checks.csv")
show_csv("Code line trace", LOGS / "code_line_trace.csv")
run_manifest = json.loads((LOGS / "run_manifest.json").read_text(encoding="utf-8"))
display(pd.DataFrame([run_manifest]))


### P4 D08 data card
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/reports/P4_D08_DATA_CARD.md`

# P4 D08 Data Card

## Status
- final_status: **P4_DATA_CONTRACT_READY**
- D08 shape: `[10242, 151]`
- D08 hash: `598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962`

## Grain And Keys
- Grain: 2024년 × outcome 학교 × outcome 캠퍼스 × outcome 학과
- Frozen P4 key: `analysis_year`, `p4_school_uid`, `p4_campus_uid`, `p4_dept_uid`, with `outcome_row_id` as stable row identity.
- P4 composite duplicate rows: `0`
- Carried headcount UID composite duplicate rows: `18`. This is tracked as WARN because `dept_uid` is a matched D01 identifier, not the outcome spine key.

## Coverage
- structure high-confidence: `8561/10242 = 83.59%`
- major high/medium: `10099/10242 = 98.60%`
- Local2 D07 hash: `f44c6f9e7a7539361dce301686144780e0ce588deb929b625ebe811674bb621f`
- D07 to D08 GOMS mismatch cells: `0`

## P4 Restrictions
- Do not treat `manual_pending` or `unmatched` rows as having observed structure context.
- Do not use ambiguous/unknown `major_group_7` rows in major/context/GOMS samples.
- Do not activate all-null D04 industry features (`ctx24_industry_hhi`, `ctx24_industry_top3_pct`).
- Do not impute, scale, one-hot encode, train models, or recalculate ratios in this contract notebook.


### P4 data integrity report
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/reports/P4_DATA_INTEGRITY_REPORT.md`

# P4 Data Integrity Report

## Final Status
**P4_DATA_CONTRACT_READY**

## Core Hashes
- D01 v2: `2f187b5af44c828d4107af12368029caf6b83b6254af75b9653b6402b8f1b0ce` shape `[34969, 186]`
- D03 v2: `c6fd569052684502e5bab5758510d3cd945f68ddeaa47fdfb3e9bab803889dca` shape `[10242, 108]`
- D08 v2: `598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962` shape `[10242, 151]`

## Gate Summary
- Manifest hash mismatch: `0`
- Manifest shape mismatch: `0`
- Manifest missing files: `0`
- D01 canonical grain duplicate: `0`
- D01 unexplained raw count delta: `0`
- D08 P4 composite duplicate: `0`
- Outcome mismatch cells D03->D08: `0`
- Structure high-confidence rate: `83.59%`
- Major high/medium rate: `98.60%`
- D07->D08 GOMS mismatch: `0`
- Split leakage: `0`
- Registry missing count: `0`
- All-null active feature sets: `0`
- FAIL checks: `0`

## Evidence Paths
- Manifest audit: `qa/manifest_integrity_audit.csv`
- D01 explanation: `reports/D01_GRAIN_EXPLANATION.md`
- Spine comparison: `qa/d03_d08_spine_comparison.csv`
- Structure matching: `qa/structure_match_integrity.csv`
- Major mapping: `qa/major7_mapping_integrity.csv`
- Local2 lineage: `qa/local2_context_lineage_integrity.csv`
- Split/sample: `qa/split_sample_integrity.csv`
- Registry: `qa/registry_integrity.csv`


### All gate QA summary
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/qa/all_gate_qa_summary.csv`

...

shape: (1, 7)


### Code line trace
`/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/p4_1_data_contract/logs/code_line_trace.csv`

,agent,script,function_or_cell,script_line_start,script_line_end
0,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,collect_manifest_artifacts,296,364
1,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,load_candidate_tables,367,386
2,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,gate1_d01_profile,397,509
3,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,gate2_spine,534,657
4,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,gate3_structure_matching,660,748
5,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,gate4_major_mapping,751,831
6,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,gate5_local2_lineage,834,900
7,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,gate6_splits_samples,918,1017
8,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,gate7_registries,1020,1075
9,P2-G4 Local Agent 1,scripts/p2_g4_1_data_contract_local1.py,final_status,1279,1300


shape: (12, 5)


,created_at,agent,script,python,platform,runtime_seconds,input_manifest,input_manifest_hash,output_root,final_status,fail_count,warn_count,metrics_path
0,2026-07-12T22:32:49,P2-G4 Local Agent 1,/home/sieg/projects-wsl/SBS_dataScience/script...,"3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]",Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...,25.722,/home/sieg/projects-wsl/SBS_dataScience/workbo...,b9ea9528b24630f15be8913012b40df81692d16e849120...,/home/sieg/projects-wsl/SBS_dataScience/workbo...,P4_DATA_CONTRACT_READY,0,1,/home/sieg/projects-wsl/SBS_dataScience/workbo...


## Final Contract Status


In [ ]:
final = json.loads((REPORTS / "final_status.json").read_text(encoding="utf-8"))
display(pd.DataFrame([
    {
        "final_status": final["final_status"],
        "manifest_hash_mismatch": final["gate0"]["hash_mismatch_n"],
        "manifest_shape_mismatch": final["gate0"]["shape_mismatch_n"],
        "d01_grain_duplicate": final["gate1"]["d01_key_duplicate_n"],
        "d08_p4_composite_duplicate": final["gate2"]["p4_composite_key_duplicate_n"],
        "structure_high_conf_rate": final["gate3"]["high_conf_match_rate"],
        "major_high_medium_rate": final["gate4"]["major_high_medium_rate"],
        "d07_d08_goms_mismatch": final["gate5"]["d07_d08_goms_mismatch_n"],
        "split_leakage": final["gate6"]["split_leakage_n"],
        "registry_missing": final["gate7"]["registry_missing_n"],
        "fail_count": final["fail_count"],
        "warn_count": final["warn_count"],
    }
]))


,final_status,manifest_hash_mismatch,manifest_shape_mismatch,d01_grain_duplicate,d08_p4_composite_duplicate,structure_high_conf_rate,major_high_medium_rate,d07_d08_goms_mismatch,split_leakage,registry_missing,fail_count,warn_count
0,P4_DATA_CONTRACT_READY,0,0,0,0,0.835872,0.986038,0,0,0,0,1
